# Stage 1 — Fine-tune Vietnamese reranker

Notebook này train `R1` từ grouped dataset do notebook mining tạo ra. Mỗi sample gồm 1 positive ở vị trí 0 và 7 hard negatives thuộc 7 documents khác nhau. Training dùng DDP thật trên Kaggle T4×2 và luôn xuất checkpoint Hugging Face.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/w4ngg/DSC-Legal-IR-QA.git"
REPO_ROOT = Path("/kaggle/working/DSC-Legal-IR-QA")
RETRIEVAL_ROOT = REPO_ROOT / "retrieval"

if not (REPO_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_ROOT)],
        check=True,
    )
else:
    print("Repository đã tồn tại, tái sử dụng:", REPO_ROOT)
SOURCE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()
assert (RETRIEVAL_ROOT / "src/legal_ir/train_reranker_stage1.py").is_file(), (
    "GitHub main chưa chứa code Stage 1. Hãy commit/push source mới trước."
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(RETRIEVAL_ROOT)],
    check=True,
)
print("Commit:", SOURCE_COMMIT)


In [ ]:
import torch

subprocess.run(["nvidia-smi", "-L"], check=True)
assert torch.cuda.is_available(), "CUDA không khả dụng"
assert torch.cuda.device_count() == 2, (
    f"Cần T4x2 nhưng hiện thấy {torch.cuda.device_count()} GPU"
)
for gpu_id in range(torch.cuda.device_count()):
    print(gpu_id, torch.cuda.get_device_name(gpu_id))

HF_HOME = Path("/kaggle/working/hf-cache")
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Disk trống:", f"{shutil.disk_usage('/kaggle/working').free / 1024**3:.2f} GiB")


## Khai báo đường dẫn

Upload ba file mining (`stage1_train.jsonl`, `stage1_manifest.json`, `stage1_skipped.jsonl`) thành Kaggle Dataset, rồi sửa `MINED_DATA_DIR`. Không trỏ vào smoke dataset.

In [ ]:
# TODO: thay đúng slug/folder của Kaggle Dataset chứa full mined data.
MINED_DATA_DIR = Path("/kaggle/input/<stage1-mined-dataset>/reranker_stage1_mined_data")
TRAIN_DATA = MINED_DATA_DIR / "stage1_train.jsonl"
DATA_MANIFEST = MINED_DATA_DIR / "stage1_manifest.json"
CONFIG = RETRIEVAL_ROOT / "configs/vietnamese_embedding_dual_long_rerank.yaml"
OUTPUT_ROOT = Path("/kaggle/working/artifacts/reranker_stage1")
SMOKE_CHECKPOINT_ROOT = OUTPUT_ROOT / "smoke_model"
CHECKPOINT_ROOT = OUTPUT_ROOT / "model"

RUN_SMOKE = True
KEEP_SMOKE_CHECKPOINT = False
OVERWRITE_FULL_OUTPUT = False
CREATE_CHECKPOINT_ZIP = False  # Bật chỉ khi /kaggle/working còn đủ disk.


In [ ]:
from legal_ir.config import PipelineConfig

for path in [TRAIN_DATA, DATA_MANIFEST, CONFIG]:
    assert path.is_file(), f"Thiếu file: {path}"
manifest = json.loads(DATA_MANIFEST.read_text(encoding="utf-8"))
assert manifest["artifact_type"] == "stage1_grouped_reranker_training_data"
assert manifest["smoke_test"] is False, "Không train model chính bằng smoke dataset"
assert manifest["dataset"]["group_size"] == 8
assert manifest["training_split"]["verified"] is True

digest = hashlib.sha256()
with TRAIN_DATA.open("rb") as handle:
    for block in iter(lambda: handle.read(1024 * 1024), b""):
        digest.update(block)
assert digest.hexdigest() == manifest["dataset"]["sha256"]

with TRAIN_DATA.open("r", encoding="utf-8") as handle:
    sample = json.loads(handle.readline())
assert len(sample["negatives"]) == 7
assert sample["positive"]["document_id"] in sample["gold_document_ids"]
assert not ({row["document_id"] for row in sample["negatives"]} & set(sample["gold_document_ids"]))

resolved = PipelineConfig.from_yaml(CONFIG)
assert resolved.reranker.model_name == "AITeamVN/Vietnamese_Reranker"
assert resolved.reranker.revision == "f536976248403314225d7fdfdbc87f0e9516a54e"
assert resolved.reranker.max_length == 2304
print("Groups:", manifest["dataset"]["group_count"])
print("Dataset SHA256:", digest.hexdigest())
print("Dataset/config validation OK")


In [ ]:
def training_command(output_dir: Path, *, max_groups: int | None = None, overwrite: bool = False):
    command = [
        sys.executable,
        "-m",
        "torch.distributed.run",
        "--standalone",
        "--nproc_per_node=2",
        "-m",
        "legal_ir.train_reranker_stage1",
        "--train-data", str(TRAIN_DATA),
        "--data-manifest", str(DATA_MANIFEST),
        "--config", str(CONFIG),
        "--output-dir", str(output_dir),
        "--epochs", "1",
        "--per-device-group-batch-size", "1",
        "--gradient-accumulation-steps", "8",
        "--learning-rate", "2e-5",
        "--weight-decay", "0.01",
        "--warmup-ratio", "0.1",
        "--mixed-precision", "fp16",
        "--gradient-checkpointing",
        "--save-steps", "250",
        "--save-total-limit", "1",
        "--log-every-steps", "10",
    ]
    if max_groups is not None:
        command.extend(["--max-groups", str(max_groups)])
    if overwrite:
        command.append("--overwrite-output-dir")
    return command

if RUN_SMOKE:
    subprocess.run(
        training_command(SMOKE_CHECKPOINT_ROOT, max_groups=32, overwrite=True),
        check=True,
        env=os.environ.copy(),
    )
    smoke_checkpoint_name = (SMOKE_CHECKPOINT_ROOT / "LAST_CHECKPOINT.txt").read_text().strip()
    smoke_checkpoint = SMOKE_CHECKPOINT_ROOT / smoke_checkpoint_name
    assert (smoke_checkpoint / "config.json").is_file()
    assert any(smoke_checkpoint.glob("*.safetensors")) or (smoke_checkpoint / "pytorch_model.bin").is_file()
    print("Smoke training OK:", smoke_checkpoint)
    if not KEEP_SMOKE_CHECKPOINT:
        shutil.rmtree(SMOKE_CHECKPOINT_ROOT)
        print("Đã xóa smoke checkpoint để giải phóng disk")


## Train Stage 1 đầy đủ

Smoke checkpoint mặc định được xóa ngay sau khi validate để giải phóng disk. Effective batch là 16 groups/update: 1 group/GPU × 2 GPU × gradient accumulation 8.

In [ ]:
subprocess.run(
    training_command(CHECKPOINT_ROOT, overwrite=OVERWRITE_FULL_OUTPUT),
    check=True,
    env=os.environ.copy(),
)


In [ ]:
checkpoint_name = (CHECKPOINT_ROOT / "LAST_CHECKPOINT.txt").read_text(encoding="utf-8").strip()
CHECKPOINT = CHECKPOINT_ROOT / checkpoint_name
assert CHECKPOINT.is_dir(), CHECKPOINT
assert (CHECKPOINT / "config.json").is_file()
assert (CHECKPOINT / "stage1_training_manifest.json").is_file()
weight_files = sorted(CHECKPOINT.glob("*.safetensors"))
if not weight_files:
    weight_files = sorted(CHECKPOINT.glob("pytorch_model*.bin"))
assert weight_files, "Checkpoint thiếu model weights"
assert any(CHECKPOINT.glob("tokenizer*")), "Checkpoint thiếu tokenizer"
checkpoint_manifest = json.loads((CHECKPOINT / "stage1_training_manifest.json").read_text(encoding="utf-8"))
assert checkpoint_manifest["training_dataset_sha256"] == manifest["dataset"]["sha256"]
print("FINAL CHECKPOINT:", CHECKPOINT)
print("Weights:", [path.name for path in weight_files])
print(json.dumps(checkpoint_manifest, ensure_ascii=False, indent=2))


In [ ]:
# Tạo config inference riêng; không sửa preset pretrained gốc.
import yaml

with CONFIG.open("r", encoding="utf-8") as handle:
    inference_config = yaml.safe_load(handle)
inference_config["reranker"]["model_name"] = str(CHECKPOINT)
inference_config["reranker"]["revision"] = None
inference_config["reranker"]["multi_gpu"] = True
R1_CONFIG = OUTPUT_ROOT / "vietnamese_embedding_dual_long_rerank_stage1.yaml"
with R1_CONFIG.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(inference_config, handle, allow_unicode=True, sort_keys=False)
print("Inference config:", R1_CONFIG)
print(R1_CONFIG.read_text(encoding="utf-8"))


In [ ]:
# Tùy chọn: ZIP checkpoint. Kaggle Save Version đã giữ folder trong /kaggle/working,
# vì vậy mặc định không ZIP để tránh cần thêm một bản sao model trên disk.
from zipfile import ZIP_STORED, ZipFile
from IPython.display import FileLink, display

if CREATE_CHECKPOINT_ZIP:
    ZIP_PATH = Path("/kaggle/working/vietnamese_reranker_stage1.zip")
    files = sorted(path for path in CHECKPOINT.rglob("*") if path.is_file())
    files.append(R1_CONFIG)
    source_bytes = sum(path.stat().st_size for path in files)
    free_bytes = shutil.disk_usage(ZIP_PATH.parent).free
    assert free_bytes > int(source_bytes * 1.05), (
        f"Không đủ disk để ZIP: input={source_bytes / 1024**3:.2f} GiB, free={free_bytes / 1024**3:.2f} GiB"
    )
    with ZipFile(ZIP_PATH, "w", compression=ZIP_STORED, allowZip64=True) as archive:
        for source in files:
            if source == R1_CONFIG:
                arcname = Path("vietnamese_reranker_stage1") / source.name
            else:
                arcname = Path("vietnamese_reranker_stage1/checkpoint") / source.relative_to(CHECKPOINT)
            archive.write(source, arcname=str(arcname))
        archive.writestr("vietnamese_reranker_stage1/SOURCE_COMMIT.txt", SOURCE_COMMIT + "\n")
    zip_digest = hashlib.sha256()
    with ZIP_PATH.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            zip_digest.update(block)
    CHECKSUM = ZIP_PATH.with_suffix(".zip.sha256")
    CHECKSUM.write_text(f"{zip_digest.hexdigest()}  {ZIP_PATH.name}\n", encoding="utf-8")
    print("Checkpoint bundle:", ZIP_PATH, f"{ZIP_PATH.stat().st_size / 1024**3:.2f} GiB")
    display(FileLink(str(ZIP_PATH)))
    display(FileLink(str(CHECKSUM)))
else:
    print("Không ZIP. Hãy Save Version để giữ folder:", CHECKPOINT_ROOT)
